### Setup

In [1]:
import boto3
import json
import requests
import feedparser
import pandas as pd

from datetime import datetime
from zoneinfo import ZoneInfo

from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_core.messages import SystemMessage, HumanMessage

### Check if conversation is possible

In [2]:
MODEL_ID = "jp.anthropic.claude-haiku-4-5-20251001-v1:0"

bedrock_runtime = boto3.Session(
    profile_name="bedrock-dev",
    region_name="ap-northeast-1"
).client("bedrock-runtime")

In [3]:
response = bedrock_runtime.converse(
    modelId=MODEL_ID,
    messages=[
        {
            "role": "user",
            "content": [
                {"text": "こんにちは。あなたのモデル名を教えてください。"}
            ]
        }
    ],
    inferenceConfig={
        "maxTokens": 200,
        "temperature": 0
    }
)

print(response["output"]["message"]["content"][0]["text"])

こんにちは。私はClaudeという名前のAIアシスタントです。Anthropicによって開発されました。

何かお手伝いできることがあれば、お気軽にお聞きください。


### Check using a weather API
#### Retrieve information using an API

In [4]:
import requests

# Retrieve location information for Fukui City
url = "https://geocoding-api.open-meteo.com/v1/search"

params = {
    "name": "Fukui",
    "count": 10,
    "language": "ja",
    "format": "json"
}

response = requests.get(url, params=params)
response.raise_for_status()

locations = response.json()["results"]


fukui = next(
    (
        location for location in locations
        if location.get("country_code") == "JP"
        and location.get("name") == "福井市"
    ),
    None
)

if fukui is None:
    raise ValueError("日本の福井市が見つかりませんでした")

latitude = fukui["latitude"]
longitude = fukui["longitude"]

print("都市:", fukui["name"])
print("緯度:", latitude)
print("経度:", longitude)

都市: 福井市
緯度: 36.06443
経度: 136.22256


#### Check what kind of information is included.

In [5]:
# Open-Meteo weather api

weather_url = "https://api.open-meteo.com/v1/forecast"

weather_params = {
    "latitude": latitude,
    "longitude": longitude,
    "current": [
        "temperature_2m",
        "apparent_temperature",
        "precipitation",
        "weather_code",
        "wind_speed_10m"
    ],
    "daily": [
        "temperature_2m_max",
        "temperature_2m_min",
        "precipitation_probability_max",
        "weather_code"
    ],
    "timezone": "Asia/Tokyo",
    "forecast_days": 1
}

response = requests.get(
    weather_url,
    params=weather_params,
    timeout=30
)

response.raise_for_status()

weather_data = response.json()

print(weather_data)

{'latitude': 36.05, 'longitude': 136.25, 'generationtime_ms': 0.15163421630859375, 'utc_offset_seconds': 32400, 'timezone': 'Asia/Tokyo', 'timezone_abbreviation': 'GMT+9', 'elevation': 18.0, 'current_units': {'time': 'iso8601', 'interval': 'seconds', 'temperature_2m': '°C', 'apparent_temperature': '°C', 'precipitation': 'mm', 'weather_code': 'wmo code', 'wind_speed_10m': 'km/h'}, 'current': {'time': '2026-11-27T13:30', 'interval': 900, 'temperature_2m': 18.7, 'apparent_temperature': 16.6, 'precipitation': 0.0, 'weather_code': 1, 'wind_speed_10m': 10.7}, 'daily_units': {'time': 'iso8601', 'temperature_2m_max': '°C', 'temperature_2m_min': '°C', 'precipitation_probability_max': '%', 'weather_code': 'wmo code'}, 'daily': {'time': ['2026-11-27'], 'temperature_2m_max': [18.8], 'temperature_2m_min': [7.9], 'precipitation_probability_max': [72], 'weather_code': [61]}}


#### Organize the necessary information.

In [6]:
current = weather_data["current"]
daily = weather_data["daily"]

weather_report = {
    "city": "福井市",
    "date": daily["time"][0],
    "temperature": current["temperature_2m"],
    "feels_like": current["apparent_temperature"],
    "wind_speed": current["wind_speed_10m"],
    "precipitation": current["precipitation"],
    "max_temperature": daily["temperature_2m_max"][0],
    "min_temperature": daily["temperature_2m_min"][0],
    "precipitation_probability": daily[
        "precipitation_probability_max"
    ][0],
    "weather_code": current["weather_code"]
}

print(weather_report)

{'city': '福井市', 'date': '2026-11-27', 'temperature': 18.7, 'feels_like': 16.6, 'wind_speed': 10.7, 'precipitation': 0.0, 'max_temperature': 18.8, 'min_temperature': 7.9, 'precipitation_probability': 72, 'weather_code': 1}


#### Quantify the weather by assigning it a score, making it easier to understand.

In [7]:
# Weather Information
weather_summary = weather_report

# Weather Code Conversion
weather_code_map = {
    0: "快晴",
    1: "晴れ",
    2: "一部曇り",
    3: "曇り",
    51: "弱い霧雨",
    53: "中程度の霧雨",
    61: "弱い雨",
    63: "中程度の雨",
    65: "強い雨"
}

weather_summary["weather"] = weather_code_map.get(
    weather_summary["weather_code"],
    "その他"
)

print(weather_summary)


{'city': '福井市', 'date': '2026-11-27', 'temperature': 18.7, 'feels_like': 16.6, 'wind_speed': 10.7, 'precipitation': 0.0, 'max_temperature': 18.8, 'min_temperature': 7.9, 'precipitation_probability': 72, 'weather_code': 1, 'weather': '晴れ'}


### Using a weather API with LangGraph
#### Setup

In [8]:
import os
import boto3

os.environ["AWS_PROFILE"] = "bedrock-dev"
AWS_REGION = "ap-northeast-1"

session = boto3.Session()

In [9]:
from langchain_aws import ChatBedrockConverse

llm = ChatBedrockConverse(
    model="jp.anthropic.claude-haiku-4-5-20251001-v1:0",
    region_name=AWS_REGION,
    credentials_profile_name="bedrock-dev",
    temperature=0
)

#### Build an AI agent that feeds weather data into an LLM to automatically generate weather analysis reports.

In [10]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# Data Handled by Agents
class WeatherState(TypedDict):
    user_input: str
    
    weather: dict
    analysis: str
    
WEATHER_SYSTEM_PROMPT = """
あなたは天気分析の専門家です。

天気情報をもとに、適切な行動を提案してください。
不明な情報は推測しないでください。
"""

WEATHER_ADVICE_PROMPT = """
    以下の天気情報を分析してください。

    ユーザーの質問:
    {user_input}

    天気情報：
    {weather}

    天気情報に存在しない情報は推測せず、
    不明であることを伝えてください。
"""

# Weather Analysis Node
def analyze_weather(state: WeatherState):

    # ① Obtain necessary information from the status.
    user_input = state["user_input"]
    weather = json.dumps(state["weather"], ensure_ascii=False, indent=2)
    
    # ② Generate a Human Message Prompt
    weather_advice_prompt = WEATHER_ADVICE_PROMPT.format(user_input=user_input, weather=weather)
    
    # ③ Execute LLM
    response = llm.invoke([
        SystemMessage(content=WEATHER_SYSTEM_PROMPT),
        HumanMessage(content=weather_advice_prompt)
    ])

    # ④ Return the analysis results to State.
    return {
        "analysis": response.content
    }

# Building with LangGraph
builder = StateGraph(WeatherState)

builder.add_node("analyze_weather", analyze_weather)

builder.add_edge(START, "analyze_weather")
builder.add_edge("analyze_weather", END)

weather_agent = builder.compile()

#### Connect the aforementioned API retrieval process to the agent execution process.

In [11]:
# Run the weather analysis agent
result = weather_agent.invoke({
    "user_input": "今日はランニングをしてもいいですか？",
    "weather": weather_summary,
    "analysis": ""
})

# Display analysis results
print(result["analysis"])
print()
print(result)

# 福井市のランニング適性分析（2026年11月27日）

## 天気情報の要点
- **天気**：晴れ
- **気温**：18.7℃（体感気温：16.6℃）
- **風速**：10.7 m/s
- **降水量**：0.0 mm
- **降水確率**：72%

## ランニング実施の判断

**ランニングは可能ですが、注意が必要です。**

### ポジティブな要素
- ✅ 現在は**晴れ**で視界が良好
- ✅ 降水量は**0.0 mm**（現在は雨が降っていない）
- ✅ 気温は**18.7℃**で、ランニングに適した温度帯

### 注意が必要な要素
- ⚠️ **降水確率が72%と高い** → 今後雨が降る可能性が高い
- ⚠️ **風速10.7 m/s**は中程度の風 → 走行時に影響がある可能性

## 推奨事項
1. **実施するなら早めに**：降水確率が高いため、できるだけ早い時間帯での実施をお勧めします
2. **雨具の準備**：降水確率が高いため、レインウェアを持参することをお勧めします
3. **風への対策**：中程度の風があるため、走行ルートや速度調整をご検討ください

**不明な情報**：湿度、紫外線指数、気圧などの詳細情報は提供されていません。

{'user_input': '今日はランニングをしてもいいですか？', 'weather': {'city': '福井市', 'date': '2026-11-27', 'temperature': 18.7, 'feels_like': 16.6, 'wind_speed': 10.7, 'precipitation': 0.0, 'max_temperature': 18.8, 'min_temperature': 7.9, 'precipitation_probability': 72, 'weather_code': 1, 'weather': '晴れ'}, 'analysis': '# 福井市のランニング適性分析（2026年11月27日）\n\n## 天気情報の要点\n- **天気**：晴れ\n- **気温**：18.7℃（体感気温：16.6℃）\n- **風速**：10.7 m/s\n- **降水量**：0.0 mm\n- **降水確率**：72%\n\n## ランニング実施の判断

### Fitness Agent
#### Definition of FitnessState

In [12]:
class FitnessState(TypedDict):

    goal: str
    available_minutes: int
    fatigue_level: list[str]
    symptoms: list[str]

    analysis: str
    safety_result: str
    workout_plan: str

#### Create initial state

In [13]:
initial_state: FitnessState = {
    "goal": "持久力向上",
    "available_minutes": 30,
    "fatigue_level": 2,
    "recent_workouts": [
        "昨日:5kmランニング",
        "一昨日:休息",
        "3日前：筋力トレーニング"
    ],
    "symptoms": [],

    "analysis": "",
    "safety_result": "",
    "workout_plan": ""
}

print(initial_state)

{'goal': '持久力向上', 'available_minutes': 30, 'fatigue_level': 2, 'recent_workouts': ['昨日:5kmランニング', '一昨日:休息', '3日前：筋力トレーニング'], 'symptoms': [], 'analysis': '', 'safety_result': '', 'workout_plan': ''}


#### Creating common functions

In [14]:
FITNESS_SYSTEM_PROMPT = """
    あなたはFitness Agentです。

    ユーザーの運動目的や体調を考慮して、
    適切な運動メニューを提案してください。

    【基本ルール】
    - 運動目的を優先する
    - 指定された時間内に収める
    - 疲労度に応じて運動強度を調整する
    - 痛みや体調不良がある場合は安全を優先する
    - 不明な情報を勝手に補完しない

    【出力ルール】
    - JSON形式のみで出力する
    - Markdownのコードブロックは使用しない
    - 指定されたJSON構造を維持する
"""

def invoke_llm(
    system_prompt: str,
    user_prompt: str,
) -> str:

    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt)
    ])

    return response.content

#### Create an Analyse Node

In [15]:
def analyze_node(state: FitnessState) -> dict:

    user_prompt = f"""
    以下のユーザー情報を分析してください。

    運動目的：{state["goal"]}
    運動可能時間；{state["available_minutes"]}分
    疲労度：{state["fatigue_level"]}/5
    直近の運動履歴：
    {state["recent_workouts"]}
    身体の症状：
    {state["symptoms"]}

    以下の観点で分析してください。

    1. ユーザーの運動目的
    2. 現在の運動状況
    3. 疲労度と運動履歴の関係
    4. 今日の運動方針
    """

    analysis = invoke_llm(
        system_prompt=FITNESS_SYSTEM_PROMPT,
        user_prompt=user_prompt
    )

    return {
        "analysis": analysis
    }

analysis_result = analyze_node(initial_state)

print(analysis_result["analysis"])

```json
{
  "analysis": {
    "user_fitness_goal": {
      "primary_objective": "持久力向上",
      "focus_area": "心肺機能の向上と運動耐性の強化"
    },
    "current_exercise_status": {
      "available_time": "30分",
      "fatigue_level": "2/5（低い）",
      "recent_history": [
        {
          "day": "昨日",
          "activity": "5kmランニング",
          "type": "有酸素運動"
        },
        {
          "day": "一昨日",
          "activity": "休息",
          "type": "回復日"
        },
        {
          "day": "3日前",
          "activity": "筋力トレーニング",
          "type": "無酸素運動"
        }
      ],
      "body_symptoms": "なし"
    },
    "fatigue_and_history_relationship": {
      "assessment": "良好な回復状態",
      "reasoning": "疲労度が低く、昨日の運動後も十分な回復が見られている。一昨日の休息が効果的に機能している。",
      "recovery_status": "適切"
    },
    "todays_exercise_plan": {
      "recommended_approach": "持久力向上を目的とした有酸素運動",
      "intensity_level": "中程度",
      "rationale": "疲労度が低く、昨日のランニングから回復している。持久力向上には継続的な有酸素刺激が効果的。",
      "suggested_activities": [
   